In [1]:
import hashlib
import os
import platform
import uuid

STUDENT_ID = 2024080210  # 改成数字

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")

[Conda 环境] exp01
[学号] 2024080210
[机器指纹] 90cf8d31534ddeec
[RUN_ID] 2024080210-90cf8d31534ddeec


# Python 正则表达式练习

In [2]:
import re

print("--- 1. 仅包含英文字母和数字 ---")
def is_alnum(s):
    return bool(re.fullmatch(r'[a-zA-Z0-9]+', s))
print(is_alnum("Hello123")) # True
print(is_alnum("Hello 123")) # False
print(is_alnum("Hello@123")) # False

print("\n--- 2. a 后面跟着零个或多个 b ---")
print(re.findall(r'ab*', "a ab abb abbb ac")) # ['a', 'ab', 'abb', 'abbb']

print("\n--- 3. a 后面跟着一个或多个 b ---")
print(re.findall(r'ab+', "a ab abb abbb ac")) # ['ab', 'abb', 'abbb']

print("\n--- 4. a 开头 b 结尾，中间任意字符 ---")
print(re.findall(r'a.*b', "axxxb a123b ac")) # ['axxxb', 'a123b']

print("\n--- 5. 删除 IP 地址中的前导零 ---")
def remove_leading_zeros(ip):
    # \b0+ 匹配连续的前导零，(\d) 匹配最后一个数字，替换为 \1
    return re.sub(r'\b0+(\d)', r'\1', ip)
print(remove_leading_zeros("192.168.001.010")) # 192.168.1.10
print(remove_leading_zeros("000.000.000.000")) # 0.0.0.0

print("\n--- 6. 空格和下划线互换 ---")
def swap_space_underscore(s):
    # 用一个临时占位符中转，或者用 lambda
    return re.sub(r'([_ ])', lambda m: '_' if m.group(0) == ' ' else ' ', s)
print(swap_space_underscore("hello world_test")) # hello_world test

print("\n--- 7. 日期格式转换 (yyyy-mm-dd -> dd-mm-yyyy) ---")
def convert_date(s):
    return re.sub(r'(\d{4})-(\d{2})-(\d{2})', r'\3-\2-\1', s)
print(convert_date("2024-05-01")) # 01-05-2024

print("\n--- 8. 找出以 a 或 e 开头的单词 ---")
print(re.findall(r'\b[ae]\w*', "apple banana elephant orange", re.IGNORECASE)) # ['apple', 'elephant', 'orange']

print("\n--- 9. 将 Road 缩写为 Rd ---")
print(re.sub(r'\bRoad\b', 'Rd', "Main Road, Cross Road")) # Main Rd, Cross Rd

print("\n--- 10. 找出长度至少为 4 个字符的单词 ---")
print(re.findall(r'\b\w{4,}\b', "I am learning regex now")) # ['learning', 'regex']

print("\n--- 11. 删除除字母和数字之外的所有字符 ---")
print(re.sub(r'[^a-zA-Z0-9]', '', "Hello@ World! 123")) # HelloWorld123

print("\n--- 12. 删除括号及括号内的内容 ---")
def remove_parentheses(s):
    return re.sub(r'\s*\([^)]*\)', '', s)
test_str = "example (.com)\nw3resource\ngithub (.com)\nstackoverflow (.com)"
print(remove_parentheses(test_str))

--- 1. 仅包含英文字母和数字 ---
True
False
False

--- 2. a 后面跟着零个或多个 b ---
['a', 'ab', 'abb', 'abbb', 'a']

--- 3. a 后面跟着一个或多个 b ---
['ab', 'abb', 'abbb']

--- 4. a 开头 b 结尾，中间任意字符 ---
['axxxb a123b']

--- 5. 删除 IP 地址中的前导零 ---
192.168.1.10
0.0.0.0

--- 6. 空格和下划线互换 ---
hello_world test

--- 7. 日期格式转换 (yyyy-mm-dd -> dd-mm-yyyy) ---
01-05-2024

--- 8. 找出以 a 或 e 开头的单词 ---
['apple', 'elephant']

--- 9. 将 Road 缩写为 Rd ---
Main Rd, Cross Rd

--- 10. 找出长度至少为 4 个字符的单词 ---
['learning', 'regex']

--- 11. 删除除字母和数字之外的所有字符 ---
HelloWorld123

--- 12. 删除括号及括号内的内容 ---
example
w3resource
github
stackoverflow


# 从网页检索气候数据

## 网页气候数据检索说明

### 1. 数据源与访问日期
- **首选源**：维基百科（如 Madison, Wisconsin），但访问时抛出 `ConnectionResetError: [WinError 10054]`，连接被远程主机强制关闭，无法获取。
- **更换源**：按作业要求，更换为可正常访问且含气候数据表格的“天气后报”网站。
- **实际 URL**：
  - 北京：http://www.tianqihoubao.com/lishi/beijing/month/202312.html
  - 上海：http://www.tianqihoubao.com/lishi/shanghai/month/202311.html
  - 广州：http://www.tianqihoubao.com/lishi/guangzhou/month/202310.html
- **访问日期**：2026-11-27

### 2. 表格结构观察（天气后报）
- 页面中 `<table>` 节点数量为 1。
- 该表格节点的直接子节点标签名包含 `thead` 和 `tbody`。
- `thead` 下的表头行包含 `<th>` 标签（如“日期”、“天气状况”）。
- `tbody` 下包含所有数据行，数据单元格使用 `<td>` 标签，第一列是日期（即行名）。
- 页面的编码通过 `r.apparent_encoding` 自动校正，避免了乱码。

### 3. 函数逻辑说明
- `retrieve_climate_table(s)`：通过判断表格内是否同时含有“日期”和“气温”来定位气候表格。网络错误会抛出异常，无表格返回 `None`。
- `list_climate_table_row_names(s)`：提取行名。兼容了行名在普通 `<td>` 中（天气后报），以及行名在 `<th scope="row">` 中（维基等标准表格）的情况。自动过滤了标题行、月份行和来源行。

### 4. 测试覆盖
- 多城市真实页面：北京、上海、广州。
- 页面成功访问但无表格：`example.com` 返回 `None`。
- 结构差异：人工构造包含 `<th scope="row">` 的表格，测试通过。

### 读取一个页面，定位表格，观察结构

#### 1 发请求、处理编码、解析

In [3]:
import requests
from bs4 import BeautifulSoup

url = "http://www.tianqihoubao.com/lishi/beijing/month/202312.html"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "http://www.tianqihoubao.com/",
}

r = requests.get(url, headers=headers, timeout=20)
r.raise_for_status()

# 关键：天气后报是 GBK 编码，requests 自动猜测常出错，必须修正
print("猜测编码:", r.encoding)
print("实际编码:", r.apparent_encoding)
r.encoding = r.apparent_encoding

soup = BeautifulSoup(r.text, "html.parser")
print("页面标题:", soup.title.get_text(strip=True) if soup.title else None)

猜测编码: utf-8
实际编码: utf-8
页面标题: 历史天气预报查询_2023年12月份北京天气记录_北京12月天气情况_天气后报


#### 2 列出所有表格，找出气候数据表格

In [4]:
# 1. 找出页面里所有的表格
tables = soup.find_all("table")
print("当前页面表格数量:", len(tables))

# 2. 循环查看每个表格的文本内容
for i, t in enumerate(tables):
    # 提取纯文本，合并多余空格
    text = " ".join(t.get_text(" ", strip=True).split())
    print(f"\n--- 表格索引 {i} ---")
    print("文本前 100 字:", text[:100])

当前页面表格数量: 1

--- 表格索引 0 ---
文本前 100 字: 日期 天气状况(白天/夜间) 最高/最低气温 风力风向(白天/夜间) 2023年12月01日 晴 / 晴 5℃ / -6℃ 无持续风向 1-3级 / 无持续风向 1-3级 2023年12月02日 晴 


#### 3 观察表格节点的结构

In [5]:
# 取出我们要的第 0 个表格
table = tables[0]

# 1. 观察表格的直接子节点（看看是 tr 还是 thead/tbody）
child_tags = [c.name for c in table.children if getattr(c, "name", None)]
print("直接子节点标签名:", set(child_tags))  # 用 set 去重看标签类型
print("直接子节点数量:", len(child_tags))

# 2. 观察前两行（第一行通常是表头，第二行是数据）
rows = table.find_all("tr")
print("\n表格总行数:", len(rows))

for i in range(2):
    cells = rows[i].find_all(["td", "th"])
    print(f"\n第 {i} 行:")
    print("  单元格标签名:", [c.name for c in cells])
    print("  单元格纯文本:", [" ".join(c.get_text(" ", strip=True).split()) for c in cells])

直接子节点标签名: {'thead', 'tbody'}
直接子节点数量: 2

表格总行数: 34

第 0 行:
  单元格标签名: ['th', 'th', 'th', 'th']
  单元格纯文本: ['日期', '天气状况(白天/夜间)', '最高/最低气温', '风力风向(白天/夜间)']

第 1 行:
  单元格标签名: ['td', 'td', 'td', 'td']
  单元格纯文本: ['2023年12月01日', '晴 / 晴', '5℃ / -6℃', '无持续风向 1-3级 / 无持续风向 1-3级']


##### 网页表格结构观察
- **实际使用的 URL**：`http://www.tianqihoubao.com/lishi/beijing/month/202312.html`
- **访问日期**：2026-11-27
- **表格节点结构说明**：
  - 页面中 `<table>` 节点数量为 1。
  - 表格节点的直接子节点标签名为 `thead` 和 `tbody`，数量为 2。
  - `<thead>` 下的第 0 行是表头，单元格标签为 `<th>`，包含“日期”、“天气状况”、“最高/最低气温”等。
  - `<tbody>` 下包含了所有的数据行，单元格标签为 `<td>`，第一列是日期（如“2023年12月01日”）。
  - 表格总行数为 34 行（包含 1 行表头和 33 行数据）。

### 写 retrieve_climate_table(s) 函数

In [6]:
import requests
from bs4 import BeautifulSoup

def _norm(tag):
    """提取文本并合并连续空白"""
    return " ".join(tag.get_text(" ", strip=True).split())

def retrieve_climate_table(s):
    """
    接收城市页面的 URL。
    找到气候数据表格返回对应的 BeautifulSoup Tag；
    成功读取但无表格返回 None；
    网络或 HTTP 错误抛出异常。
    """
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "http://www.tianqihoubao.com/",
    }
    
    # 发请求，如果 403/404/429 等会在这里抛出异常
    r = requests.get(s, headers=headers, timeout=20)
    r.raise_for_status()
    
    # 修正编码
    r.encoding = r.apparent_encoding
    
    soup = BeautifulSoup(r.text, "html.parser")
    
    # 遍历页面中所有表格寻找气候表格
    for table in soup.find_all("table"):
        text = _norm(table)
        # 找同时包含“日期”和“气温”关键词的表格
        if "日期" in text and "气温" in text:
            return table
            
    # 页面成功获取，但没有找到符合条件的表格
    return None

In [7]:
urls = [
    "http://www.tianqihoubao.com/lishi/beijing/month/202312.html",
    "http://www.tianqihoubao.com/lishi/shanghai/month/202311.html",
    "http://www.tianqihoubao.com/lishi/guangzhou/month/202310.html",
]

for u in urls:
    table_obj = retrieve_climate_table(u)
    print(f"{u}")
    if table_obj:
        print("  => 成功找到表格！返回类型:", type(table_obj))
    else:
        print("  => 没找到表格。")

# 测试成功读取但无表格的页面
print("\n测试无表格页面 (example.com):")
no_table_result = retrieve_climate_table("https://example.com")
print("  返回结果:", no_table_result)
assert no_table_result is None
print("  => 成功返回 None")

http://www.tianqihoubao.com/lishi/beijing/month/202312.html
  => 成功找到表格！返回类型: <class 'bs4.element.Tag'>
http://www.tianqihoubao.com/lishi/shanghai/month/202311.html
  => 成功找到表格！返回类型: <class 'bs4.element.Tag'>
http://www.tianqihoubao.com/lishi/guangzhou/month/202310.html
  => 成功找到表格！返回类型: <class 'bs4.element.Tag'>

测试无表格页面 (example.com):
  返回结果: None
  => 成功返回 None


### 提取行名

In [8]:
def list_climate_table_row_names(s):
    """
    接收城市页面 URL。
    返回气候数据表格的行名（字符串列表）。
    如果没有找到表格，返回 None。
    """
    table = retrieve_climate_table(s)
    if table is None:
        return None
        
    names = []
    
    # 找出所有的行
    for tr in table.find_all("tr"):
        # 查找当前行所有的单元格（包括 td 和 th）
        cells = tr.find_all(["td", "th"])
        if not cells:
            continue
            
        # 观察发现：表头行只有 th，没有 td，所以我们跳过没有 td 的行
        tds = tr.find_all("td")
        if not tds:
            continue
            
        # 取第一个数据单元格（td）的内容作为行名
        name = _norm(tds[0])
        
        # 过滤掉可能的空行或无效行
        if not name:
            continue
            
        names.append(name)
        
    return names

# ================= 测试代码 =================
print("=== 测试 list_climate_table_row_names ===")

for u in urls:
    row_names = list_climate_table_row_names(u)
    print(f"\nURL: {u}")
    if row_names is None:
        print("  => 无表格")
    else:
        print(f"  => 成功提取到 {len(row_names)} 个行名")
        print(f"  => 前 5 个: {row_names[:5]}")
        print(f"  => 后 2 个: {row_names[-2:]}")
        
        # 确保每一个都是字符串
        assert all(isinstance(x, str) for x in row_names)
        # 确保不包含表头
        assert not any("日期" in x for x in row_names)

print("\n测试无表格页面的行名:")
assert list_climate_table_row_names("https://example.com") is None
print("  => 成功返回 None")
print("\n=== 所有测试通过！ ===")

=== 测试 list_climate_table_row_names ===

URL: http://www.tianqihoubao.com/lishi/beijing/month/202312.html
  => 成功提取到 31 个行名
  => 前 5 个: ['2023年12月01日', '2023年12月02日', '2023年12月03日', '2023年12月04日', '2023年12月05日']
  => 后 2 个: ['2023年12月30日', '2023年12月31日']

URL: http://www.tianqihoubao.com/lishi/shanghai/month/202311.html
  => 成功提取到 30 个行名
  => 前 5 个: ['2023年11月01日', '2023年11月02日', '2023年11月03日', '2023年11月04日', '2023年11月05日']
  => 后 2 个: ['2023年11月29日', '2023年11月30日']

URL: http://www.tianqihoubao.com/lishi/guangzhou/month/202310.html
  => 成功提取到 30 个行名
  => 前 5 个: ['2023年10月01日', '2023年10月02日', '2023年10月03日', '2023年10月04日', '2023年10月05日']
  => 后 2 个: ['2023年10月30日', '2023年10月31日']

测试无表格页面的行名:
  => 成功返回 None

=== 所有测试通过！ ===


In [9]:
def list_climate_table_row_names(s):
    """
    接收城市页面 URL。
    返回气候数据表格的行名（字符串列表）。
    如果没有找到表格，返回 None。
    升级版：兼容行名在 <td> 或 <th scope="row"> 的表格结构。
    """
    table = retrieve_climate_table(s)
    if table is None:
        return None
        
    names = []
    
    for tr in table.find_all("tr"):
        cells = tr.find_all(["td", "th"])
        if not cells:
            continue
            
        # 优先寻找 th scope="row"，其次找第一个 th，最后回退到第一个 td
        ths = tr.find_all("th")
        candidate = None
        for th in ths:
            if th.get("scope") == "row":
                candidate = th
                break
        if candidate is None and ths:
            candidate = ths[0]
        if candidate is None:
            candidate = cells[0]

        text = _norm(candidate)
        if not text:
            continue
            
        low = text.lower()
        
        # 过滤掉无关的行
        if "日期" in text:                              # 天气后报表头
            continue
        if text.startswith("Climate data for"):         # 维基标题行
            continue
        if low.startswith("source") or "来源" in text:  # 数据来源行
            continue
        # 过滤掉月份表头
        if low in {"month", "jan", "feb", "mar", "apr", "may", "jun", 
                   "jul", "aug", "sep", "oct", "nov", "dec"}:
            continue
            
        names.append(text)
        
    return names

In [10]:
print("=== 1. 回归测试：真实网页 ===")
for u in urls:
    names = list_climate_table_row_names(u)
    print(f"{u} => 提取到 {len(names)} 个行名，前2个: {names[:2]}")

print("\n=== 2. 结构差异测试：人工构造带 <th scope=\"row\"> 的表格 ===")
html_diff = """
<html><body>
<table>
  <caption>Climate data for Test City</caption>
  <tr><th>Month</th><th>Jan</th><th>Feb</th></tr>
  <tr><th scope="row">Average high °C</th><td>1</td><td>2</td></tr>
  <tr><th scope="row">Average low °C</th><td>-5</td><td>-4</td></tr>
  <tr><td colspan="3">Source: Test</td></tr>
</table>
</body></html>
"""

# 为了测试人工构造的 HTML，我们用 BeautifulSoup 解析，并手动跑一遍提取逻辑
soup_diff = BeautifulSoup(html_diff, "html.parser")
table_diff = soup_diff.find("table")

# 临时替换环境来测 diff_table
orig_retrieve = globals()['retrieve_climate_table']
globals()['retrieve_climate_table'] = lambda _: table_diff
diff_names = list_climate_table_row_names("dummy_url")
globals()['retrieve_climate_table'] = orig_retrieve

print("提取出的行名:", diff_names)
assert diff_names == ["Average high °C", "Average low °C"], "结构差异提取失败"
print("=> 结构差异测试通过！完美兼容 th scope='row'")

=== 1. 回归测试：真实网页 ===
http://www.tianqihoubao.com/lishi/beijing/month/202312.html => 提取到 31 个行名，前2个: ['2023年12月01日', '2023年12月02日']
http://www.tianqihoubao.com/lishi/shanghai/month/202311.html => 提取到 30 个行名，前2个: ['2023年11月01日', '2023年11月02日']
http://www.tianqihoubao.com/lishi/guangzhou/month/202310.html => 提取到 30 个行名，前2个: ['2023年10月01日', '2023年10月02日']

=== 2. 结构差异测试：人工构造带 <th scope="row"> 的表格 ===
提取出的行名: ['Average high °C', 'Average low °C']
=> 结构差异测试通过！完美兼容 th scope='row'
